# 03 · `pyproject.toml`: The Complete Guide

**Module 08 – Modules, Packaging & Dependency Management** · Estimated time: 60 min

> ❗ Not covered by the 📗 bootcamp or the 📘 handbook (which only shows `requirements.txt`). `pyproject.toml` is now **the** standard configuration file for every Python project, whether you use pip, uv, Poetry, Hatch or PDM.

### 🎯 What you will learn
- The TOML format in 5 minutes
- The three standard tables: `[build-system]`, `[project]`, `[dependency-groups]`
- **Every** `[project]` field: name, version, dynamic, description, readme, requires-python, license (PEP 639), authors, keywords, classifiers, urls, dependencies, optional-dependencies (extras), scripts, gui-scripts, entry-points
- Dependency specifiers: versions (PEP 440), extras, markers (PEP 508), direct references
- Build backends compared: setuptools, hatchling, flit-core, pdm-backend, poetry-core, **uv_build**, maturin, scikit-build-core
- `[tool.*]` sections for ruff, mypy, pytest, coverage, uv, poetry, setuptools, hatch
- Inline script metadata (PEP 723)
- Inspecting and validating requirements with the `packaging` library

## 1. TOML in 5 minutes

TOML (*Tom's Obvious Minimal Language*) is a config format with real types:

```toml
# comment
name = "demo"              # string
version = 3                # integer
pi = 3.14                  # float
enabled = true             # boolean
released = 2026-10-09      # date
tags = ["a", "b"]          # array
author = { name = "Asha", email = "asha@example.com" }   # inline table

[server]                   # table (section) → a dict
host = "localhost"

[server.tls]               # nested table → server["tls"]
cert = "cert.pem"

[[plugins]]                # array of tables → a list of dicts
name = "auth"
[[plugins]]
name = "cache"
```

In [1]:
import tomllib
doc = tomllib.loads('''
name = "demo"
released = 2026-10-09
tags = ["a", "b"]
author = { name = "Asha", email = "asha@example.com" }
[server]
host = "localhost"
[server.tls]
cert = "cert.pem"
[[plugins]]
name = "auth"
[[plugins]]
name = "cache"
''')
print(doc)

{'name': 'demo', 'released': datetime.date(2026, 10, 9), 'tags': ['a', 'b'], 'author': {'name': 'Asha', 'email': 'asha@example.com'}, 'server': {'host': 'localhost', 'tls': {'cert': 'cert.pem'}}, 'plugins': [{'name': 'auth'}, {'name': 'cache'}]}


## 2. A complete, annotated `pyproject.toml`

```toml
# ─────────────────────────── 1. HOW TO BUILD (PEP 517/518) ───────────────────────────
[build-system]
requires = ["hatchling>=1.27"]          # packages needed to BUILD the project
build-backend = "hatchling.build"       # the object a frontend (pip/uv/build) calls

# ─────────────────────────── 2. PROJECT METADATA (PEP 621) ───────────────────────────
[project]
name = "weather-cli"                    # distribution name on PyPI (normalised: weather_cli == Weather.CLI)
version = "1.2.0"                       # PEP 440 version (or list "version" in `dynamic`)
description = "Fetch the weather from your terminal"
readme = "README.md"                    # or { file = "README.rst", content-type = "text/x-rst" }
requires-python = ">=3.12"              # installers refuse other Pythons
license = "MIT"                         # PEP 639: an SPDX expression, e.g. "MIT OR Apache-2.0"
license-files = ["LICENSE*"]            # PEP 639: files to ship
authors = [{ name = "Asha Rao", email = "asha@example.com" }]
maintainers = [{ name = "Ravi K" }]
keywords = ["weather", "cli"]
classifiers = [                         # https://pypi.org/classifiers/
    "Development Status :: 4 - Beta",
    "Programming Language :: Python :: 3.15",
    "Operating System :: OS Independent",
    "Typing :: Typed",
]
dependencies = [                        # RUNTIME dependencies (PEP 508 strings)
    "httpx>=0.27,<1",
    "rich>=13",
    "tomli>=2; python_version < '3.11'",
]

[project.optional-dependencies]         # "extras": pip install "weather-cli[yaml]"
yaml = ["pyyaml>=6"]
all = ["weather-cli[yaml]"]             # an extra can include other extras

[project.urls]
Homepage = "https://github.com/asha/weather-cli"
Documentation = "https://weather-cli.readthedocs.io"
Issues = "https://github.com/asha/weather-cli/issues"
Changelog = "https://github.com/asha/weather-cli/blob/main/CHANGELOG.md"

[project.scripts]                       # console commands created on install
weather = "weather_cli.main:app"        # "command = module:function"

[project.gui-scripts]                   # like scripts, but no console window on Windows
weather-gui = "weather_cli.gui:run"

[project.entry-points."weather_cli.plugins"]   # plugin registry other packages can discover
openmeteo = "weather_cli.providers.openmeteo:Provider"

# ─────────────────────────── 3. DEV DEPENDENCY GROUPS (PEP 735) ─────────────────────
[dependency-groups]                     # NOT installed for end users; tools install them for you
test = ["pytest>=8", "pytest-cov"]
lint = ["ruff", "mypy"]
dev = [{ include-group = "test" }, { include-group = "lint" }, "ipython"]

# ─────────────────────────── 4. TOOL SETTINGS ([tool.<name>]) ────────────────────────
[tool.hatch.build.targets.wheel]
packages = ["src/weather_cli"]

[tool.ruff]
line-length = 100
target-version = "py312"
[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]

[tool.mypy]
strict = true

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra --strict-markers"

[tool.coverage.run]
branch = true
source = ["weather_cli"]
```

## 3. `[build-system]` and build backends

When a tool builds your project (to make a wheel, or to install from source), it:
1. creates an isolated environment
2. installs `build-system.requires`
3. calls the `build-backend` object (`build_wheel()`, `build_sdist()`; PEP 517)

| Backend | `requires` / `build-backend` | Notes |
|---|---|---|
| **setuptools** | `["setuptools>=77"]` / `setuptools.build_meta` | the classic; C extensions; huge ecosystem |
| **hatchling** | `["hatchling"]` / `hatchling.build` | modern, simple, popular; plugins (e.g. version from Git) |
| **flit-core** | `["flit_core>=3.11,<4"]` / `flit_core.buildapi` | minimal, pure-Python packages |
| **pdm-backend** | `["pdm-backend"]` / `pdm.backend` | used by PDM |
| **poetry-core** | `["poetry-core>=2.0"]` / `poetry.core.masonry.api` | used by Poetry |
| **uv_build** | `["uv_build>=0.13,<0.14"]` / `uv_build` | 🆕 uv's own very fast backend (pure-Python packages) |
| **maturin** | `["maturin>=1.8"]` / `maturin` | Rust extensions (PyO3) |
| **scikit-build-core** | `["scikit-build-core"]` / `scikit_build_core.build` | C/C++ via CMake |

> 💡 The backend is independent of the **frontend/installer**. `pip`, `uv`, `python -m build` and `poetry` can all build any project, whichever backend it declares.

## 4. The `[project]` table: field by field

| Field | Required? | Notes |
|---|---|---|
| `name` | ✅ | letters, digits, `-`, `_`, `.`; compared case-/separator-insensitively |
| `version` | ✅ (or `dynamic`) | PEP 440 |
| `dynamic` | | fields filled in by the backend, e.g. `["version"]` from Git tags or `__version__` |
| `description` | | one line |
| `readme` | | path, or `{file=..., content-type=...}`; becomes the PyPI page |
| `requires-python` | | e.g. `">=3.12"`, always set it |
| `license` | | 🆕 PEP 639: SPDX string like `"MIT"`, `"Apache-2.0"`, `"MIT OR GPL-3.0-or-later"` |
| `license-files` | | glob patterns of license files to include |
| `authors` / `maintainers` | | list of `{name, email}` |
| `keywords` | | list of strings |
| `classifiers` | | Trove classifiers |
| `urls` | | label → URL |
| `dependencies` | | PEP 508 strings |
| `optional-dependencies` | | extras: name → list |
| `scripts` / `gui-scripts` | | command → `module:function` |
| `entry-points` | | group → {name → object} |

> ⚠️ **Deprecated license style:** `license = { text = "MIT" }` / `license = { file = "LICENSE" }` and `License :: OSI Approved :: ...` classifiers are superseded by PEP 639's SPDX `license` string + `license-files`.

### `dynamic` version example (hatchling)

```toml
[project]
name = "mypkg"
dynamic = ["version"]

[tool.hatch.version]
path = "src/mypkg/__init__.py"     # reads __version__ = "1.4.0"
```

## 5. Dependency specifiers in depth

A dependency string is: `name [extras] (version specifiers) ; environment markers` or `name @ URL`.

In [2]:
from packaging.requirements import Requirement     # `packaging` is the reference implementation used by pip/uv

for text in [
    "requests>=2.31,<3",
    "httpx[http2,socks]~=0.28",
    "tomli>=2; python_version < '3.11'",
    "pywin32>=306; sys_platform == 'win32'",
    "mylib @ git+https://github.com/me/mylib.git@v1.0",
]:
    r = Requirement(text)
    print(f"{r.name:<9} extras={sorted(r.extras)!s:<18} spec={str(r.specifier)!r:<14} "
          f"marker={str(r.marker) if r.marker else '-':<28} url={r.url or '-'}")

requests  extras=[]                 spec='<3,>=2.31'    marker=-                            url=-
httpx     extras=['http2', 'socks'] spec='~=0.28'       marker=-                            url=-
tomli     extras=[]                 spec='>=2'          marker=python_version < "3.11"      url=-
pywin32   extras=[]                 spec='>=306'        marker=sys_platform == "win32"      url=-
mylib     extras=[]                 spec=''             marker=-                            url=git+https://github.com/me/mylib.git@v1.0


### Versions (PEP 440)

`1.2.0`, `2.0.0a1` (alpha), `2.0.0b2` (beta), `2.0.0rc1` (release candidate), `1.2.0.post1` (post-release), `1.3.0.dev4` (dev), `1.2.0+local.7` (local; not allowed on PyPI), epochs `1!2.0`.

In [3]:
from packaging.version import Version
from packaging.specifiers import SpecifierSet

versions = ["1.9", "2.0.0a1", "2.0.0rc1", "2.0", "2.0.1", "2.1.0.dev3", "2.1", "3.0"]
print(sorted(versions, key=Version))                       # correct ordering (string sorting would be wrong!)
spec = SpecifierSet(">=2.0,<3")
print(list(spec.filter(versions)))                         # pre-releases excluded by default
print(list(SpecifierSet("~=2.0").filter(versions)))        # compatible release: >=2.0, ==2.*
print(Version("2.0.0rc1").is_prerelease, Version("1.2.0.post1").is_postrelease)

['1.9', '2.0.0a1', '2.0.0rc1', '2.0', '2.0.1', '2.1.0.dev3', '2.1', '3.0']
['2.0', '2.0.1', '2.1']
['2.0', '2.0.1', '2.1']
True True


| Operator | Meaning | Example |
|---|---|---|
| `==` | equal (wildcards allowed) | `==2.*` |
| `!=` | not equal | `!=2.0.1` |
| `>=`, `<=`, `>`, `<` | comparisons | `>=2,<3` |
| `~=` | compatible release | `~=2.2` → `>=2.2,==2.*`; `~=2.2.1` → `>=2.2.1,==2.2.*` |
| `===` | arbitrary string equality (rare) | |

> 💡 **Libraries** should use *ranges* (`>=2.31,<3`) so they combine well with other packages. **Applications** pin *exact* versions in a **lock file** (`uv.lock`, `poetry.lock`, `pylock.toml`) rather than in `pyproject.toml`.

### Environment markers

In [4]:
from packaging.markers import Marker
for m in ["python_version >= '3.12'", "sys_platform == 'win32'",
          "platform_machine == 'arm64' and os_name == 'posix'", "implementation_name == 'pypy'"]:
    print(f"{m:<55} → {Marker(m).evaluate()}")

python_version >= '3.12'                                → True
sys_platform == 'win32'                                 → False
platform_machine == 'arm64' and os_name == 'posix'      → True
implementation_name == 'pypy'                           → False


Available markers include `python_version`, `python_full_version`, `sys_platform`, `platform_system`, `platform_machine`, `os_name`, `implementation_name`, `extra`.

## 6. Extras vs dependency groups

| | `[project.optional-dependencies]` (extras) | `[dependency-groups]` (PEP 735) |
|---|---|---|
| For whom? | **users** of your package | **developers** of your package |
| Published to PyPI? | ✅ (part of metadata) | ❌ |
| Installed with | `pip install "pkg[yaml]"` | `uv sync --group lint`, `pip install --group lint` (pip ≥ 25.1) |
| Typical content | optional features (`yaml`, `postgres`, `all`) | `test`, `lint`, `docs`, `dev` |

> ⚠️ **Older conventions:** dev dependencies hidden in extras (`[project.optional-dependencies] dev = [...]`), `[tool.uv] dev-dependencies = [...]`, or Poetry's `[tool.poetry.dev-dependencies]` (deprecated in favour of `[tool.poetry.group.dev.dependencies]`). PEP 735 `[dependency-groups]` is now the standard.

## 7. Entry points: commands and plugins

`[project.scripts]` creates executables on install:
```toml
[project.scripts]
weather = "weather_cli.main:app"     # running `weather` calls weather_cli.main.app()
```
`[project.entry-points."group"]` lets *other* packages register plugins your code discovers at runtime:

In [5]:
from importlib.metadata import entry_points
eps = entry_points(group="console_scripts")
print(len(eps), "console scripts installed, e.g.:", sorted(ep.name for ep in eps)[:8])
pytest_plugins = entry_points(group="pytest11")              # how pytest finds its plugins
print([ep.name for ep in pytest_plugins])

71 console scripts installed, e.g.: ['cffi-gen-src', 'coverage', 'coverage-3.15', 'coverage3', 'debugpy', 'debugpy-adapter', 'dmypy', 'docutils']
['pytest_cov', 'asyncio', 'hypothesispytest', 'pytest_mock', 'anyio', 'platformdirs']


## 8. `[tool.*]`: configuration for every tool in one file

| Section | Tool |
|---|---|
| `[tool.ruff]`, `[tool.ruff.lint]`, `[tool.ruff.format]` | Ruff linter/formatter |
| `[tool.mypy]`, `[tool.pyright]` | type checkers |
| `[tool.pytest.ini_options]` | pytest |
| `[tool.coverage.run]`, `[tool.coverage.report]` | coverage.py |
| `[tool.uv]`, `[tool.uv.sources]`, `[[tool.uv.index]]` | uv (lesson 04) |
| `[tool.poetry]`, `[tool.poetry.group.*]` | Poetry (lesson 05) |
| `[tool.hatch.*]`, `[tool.setuptools.*]`, `[tool.pdm.*]` | build backends |
| `[tool.black]`, `[tool.isort]` | older formatters (largely replaced by Ruff) |

### setuptools package discovery & data files

```toml
[tool.setuptools.packages.find]
where = ["src"]                      # src layout

[tool.setuptools.package-data]
mypkg = ["templates/*.html", "py.typed"]
```

## 9. Project layouts

```
flat layout                        src layout ✅ (recommended)
my-project/                        my-project/
├── pyproject.toml                 ├── pyproject.toml
├── mypkg/                         ├── src/
│   ├── __init__.py                │   └── mypkg/
│   └── core.py                    │       ├── __init__.py
└── tests/                         │       ├── core.py
                                   │       └── py.typed      ← marks the package as typed (PEP 561)
                                   ├── tests/
                                   ├── README.md
                                   └── LICENSE
```
The **src layout** prevents tests from accidentally importing the local folder instead of the *installed* package, so you catch packaging mistakes early. uv, Poetry and Hatch create it by default.

## 10. Inline script metadata (PEP 723): `pyproject` for single files

Scripts can declare their own dependencies in a special comment block:

In [6]:
script = '''# /// script
# requires-python = ">=3.12"
# dependencies = [
#   "httpx>=0.27",
#   "rich",
# ]
# ///
import httpx
from rich import print
print(httpx.get("https://httpbin.org/uuid", timeout=10).json())
'''
import re, tomllib
block = re.search(r"(?m)^# /// script$\s(?P<content>(^#(| .*)$\s)+)^# ///$", script)
metadata = tomllib.loads("".join(line[2:] for line in block["content"].splitlines(keepends=True)))
print(metadata)

{'requires-python': '>=3.12', 'dependencies': ['httpx>=0.27', 'rich']}


Run it with `uv run script.py` or `pipx run script.py`, and the tool creates a temporary environment with those dependencies (lesson 04).

## 11. Reading your own project's metadata at runtime

In [7]:
from importlib.metadata import version, metadata, requires, distribution
print(version("requests"))
md = metadata("requests")
print(md["Summary"], "|", md["Requires-Python"])
print(requires("requests"))
print(distribution("requests").read_text("WHEEL").splitlines()[:3])

2.34.2
Python HTTP for Humans. | >=3.10
['charset_normalizer<4,>=2', 'idna<4,>=2.5', 'urllib3<3,>=1.26', 'certifi>=2023.5.7', 'PySocks!=1.5.7,>=1.5.6; extra == "socks"', 'chardet<8,>=3.0.2; extra == "use-chardet-on-py3"']
['Wheel-Version: 1.0', 'Generator: setuptools (82.0.1)', 'Root-Is-Purelib: true']


✅ To expose your package's version, prefer `importlib.metadata.version("your-dist-name")` over hard-coding `__version__` in two places.

## 12. Legacy files and what replaced them

| ⚠️ Old | ✅ Now |
|---|---|
| `setup.py` with `setup(...)` metadata | `[project]` in `pyproject.toml` (setup.py only for custom build logic) |
| `setup.cfg` `[metadata]` / `[options]` | `[project]` |
| `requirements-dev.txt` | `[dependency-groups]` |
| `MANIFEST.in` | still used by setuptools sdists; other backends use their own include/exclude settings |
| `tox.ini`, `pytest.ini`, `.coveragerc`, `mypy.ini`, `.flake8` | `[tool.*]` sections (most tools) |
| `Pipfile` / `Pipfile.lock` (Pipenv) | `pyproject.toml` + `uv.lock` / `poetry.lock` |
| `python setup.py sdist bdist_wheel upload` | `python -m build` + `twine upload`, or `uv build` + `uv publish` |

## 📝 Summary
- `[build-system]` says *how to build*; `[project]` says *what the project is*; `[dependency-groups]` holds dev tools; `[tool.*]` configures tools.
- Dependencies are PEP 508 strings: name, extras, version specifiers (PEP 440), markers, URLs.
- Libraries use ranges; applications rely on lock files.
- Extras are for users; dependency groups are for developers.
- Use `license = "SPDX"` (PEP 639), the src layout, and PEP 723 metadata for single scripts.

## 🏋️ Practice
1. Write a `pyproject.toml` for a library `textstats` (src layout, hatchling) depending on `regex>=2024` only on Python < 3.13, with an extra `cli = ["rich"]`, a command `textstats = "textstats.cli:main"`, and dependency groups `test` and `lint`.
2. Using `packaging`, check whether `3.15.0` satisfies `>=3.10,!=3.12.*`.
3. Sort `["1.0.post1", "1.0", "1.0rc1", "1.0.dev0", "1.1a1"]` correctly.

<details><summary>Show solutions</summary>

```python
from packaging.specifiers import SpecifierSet
from packaging.version import Version
print(SpecifierSet(">=3.10,!=3.12.*").contains("3.15.0"))
print(sorted(["1.0.post1", "1.0", "1.0rc1", "1.0.dev0", "1.1a1"], key=Version))
# ['1.0.dev0', '1.0rc1', '1.0', '1.0.post1', '1.1a1']
```

```toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "textstats"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = ["regex>=2024; python_version < '3.13'"]

[project.optional-dependencies]
cli = ["rich"]

[project.scripts]
textstats = "textstats.cli:main"

[dependency-groups]
test = ["pytest"]
lint = ["ruff", "mypy"]
```
</details>